> **Credit:** This project is inspired by the free lesson
> [*Building a KPI Dashboard with Pandas*](https://www.mathewkanalytics.com/lessons/mastering-pandas/058-building-a-kpi-dashboard-with-pandas/)
> by **[Mathew K Analytics](https://www.mathewkanalytics.com/)**
> ([YouTube video](https://www.youtube.com/watch?v=lid1q679cs0)).
> The original uses the Titanic dataset; this version applies the same workflow to a retail sales dataset and extends it.

# 📊 Retail Sales KPI Dashboard with Pandas

**Goal:** Turn 8,399 raw order lines from a retail store ("Superstore", Canada, 2009–2012) into a one-page KPI dashboard
that a manager could read in 30 seconds.

**What is a KPI?** A *Key Performance Indicator* is a single number that tells you whether the business is doing well
(e.g. total sales, profit margin, average order value). A dashboard puts the most important KPIs in one place.

### The workflow (same steps the original lesson teaches)
1. **Load** the data
2. **Explore** it - what columns, types, sizes?
3. **Clean** it - fix types, fill missing values
4. **Create KPI columns** - new columns derived from existing ones
5. **Aggregate** - headline KPIs and KPIs per group
6. **Time-based KPIs** - how do things change month to month?
7. **Visualize** - build the dashboard image
8. **Make it reusable** - a function that builds a KPI report for any filter
9. **Write insights** - what does the data actually say?

### How to use this notebook
- Cells marked **✏️ Your turn** have `...` placeholders. Replace them with code.
- Stuck? Open the **💡 Hint** below the exercise. Still stuck? Look in `solutions/`.
- Run the **✅ Check** cell after each exercise. If it prints ✅ you got it right.

## Step 0 - Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:,.2f}".format)  # nicer number printing

def check(name, value, expected, tol=0.01):
    """Tiny helper that tells you if your answer is right."""
    try:
        value = float(value)
        ok = abs(value - expected) <= tol * max(1, abs(expected))
        shown = f"{value:,.4g}"
    except (TypeError, ValueError):  # e.g. you still have ... in your code
        ok, shown = False, repr(value)
    print(("✅ " if ok else "❌ ") + f"{name}: got {shown}" + ("" if ok else f" (expected about {expected:,.4g})"))

## Step 1 - Load the data

The CSV is stored in `data/`. If it's missing, we download it from the source.

**Lesson:** Not every file is UTF-8. This one contains characters like `¨` encoded as *latin-1*,
so `pd.read_csv` fails without `encoding="latin-1"`. Try removing it and see the error!

In [ ]:
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/curran/data/gh-pages/superstoreSales/superstoreSales.csv"
ROOT = Path("..") if Path("../data").exists() else Path(".")  # project folder
DATA_PATH = ROOT / "data" / "superstore_sales.csv"

source = DATA_PATH if DATA_PATH.exists() else DATA_URL
raw = pd.read_csv(source, encoding="latin-1")
df = raw.copy()  # keep the raw data untouched, work on a copy
print("Loaded from:", source)

## Step 2 - Explore

Before calculating anything, look at the data. Answer for yourself: how many rows? Which columns are numbers,
which are text, which should be dates?

**✏️ Your turn:** Print the shape of `df`, show the first 5 rows, and show `df.info()`.

In [ ]:
# ✏️ YOUR TURN - replace every ... with code
# How many rows and columns?
...

# First 5 rows
...

<details><summary>💡 Hint</summary>

`df.shape` is an attribute (no brackets). `df.head()` is a method.

</details>

In [ ]:
df.info()

**Notice:**
- `Order Date` and `Ship Date` are `object` (text), not dates → we must convert them.
- `Product Base Margin` has fewer non-null values than the other columns → missing values.
- One row is one **order line** (one product in an order). One order can have several lines - so
  "number of orders" is **not** the number of rows!

In [ ]:
df.describe()

## Step 3 - Clean

### 3a. Missing values
**✏️ Your turn:** Count the missing values in every column.

In [ ]:
# ✏️ YOUR TURN - replace every ... with code
missing = ...
missing[missing > 0]

<details><summary>💡 Hint</summary>

`df.isnull()` gives True/False for every cell; `.sum()` counts the Trues per column.

</details>

Only `Product Base Margin` has gaps. The Titanic lesson filled missing ages with the **overall median**.
We can do better: margins differ a lot between product types, so we fill each gap with the median
**of its own Product Sub-Category**. `groupby(...).transform("median")` returns a value for every row, aligned to `df`.

In [ ]:
group_median = df.groupby("Product Sub-Category")["Product Base Margin"].transform("median")
df["Product Base Margin"] = df["Product Base Margin"].fillna(group_median)
print("Missing values left:", df.isnull().sum().sum())

### 3b. Fix data types
**✏️ Your turn:** Convert `Order Date` and `Ship Date` to real dates with `pd.to_datetime`.
The format is month/day/year, e.g. `10/13/2010`.

In [ ]:
# ✏️ YOUR TURN - replace every ... with code
df["Order Date"] = ...
df["Ship Date"] = ...
df[["Order Date", "Ship Date"]].dtypes

<details><summary>💡 Hint</summary>

`pd.to_datetime(df["Order Date"], format="%m/%d/%Y")` - `%m` month, `%d` day, `%Y` 4-digit year.

</details>

In [ ]:
check("Order Date is datetime", int(pd.api.types.is_datetime64_any_dtype(df["Order Date"])), 1)
check("First order date year", df["Order Date"].min().year, 2009)

Text columns with few distinct values are *categories*. Converting them saves memory and makes intent clear (same as the original lesson):

In [ ]:
for col in ["Order Priority", "Ship Mode", "Region", "Customer Segment", "Product Category"]:
    df[col] = df[col].astype("category")
print("Duplicate rows:", df.duplicated().sum())

## Step 4 - Create KPI columns

Good KPIs often need columns that don't exist yet. We'll derive them from existing columns.

**✏️ Your turn:** Create these four columns:
| Column | Meaning |
|---|---|
| `Days to Ship` | number of days between `Order Date` and `Ship Date` |
| `Is Loss` | `True` if `Profit` is below 0 |
| `Year` | year of the order |
| `Month` | first day of the order's month (for monthly trends) - given |

In [ ]:
# ✏️ YOUR TURN - replace every ... with code
df["Days to Ship"] = ...
df["Is Loss"] = ...
df["Year"] = ...
df["Month"] = df["Order Date"].dt.to_period("M").dt.to_timestamp()

<details><summary>💡 Hint</summary>

Subtracting two date columns gives a *timedelta*; `.dt.days` turns it into a number. Date parts live under `.dt` (e.g. `.dt.year`).

</details>

In [ ]:
check("Average days to ship", df["Days to Ship"].mean(), 2.033)
check("Share of loss-making order lines", df["Is Loss"].mean(), 0.5077)
check("Number of distinct years", df["Year"].nunique(), 4)

## Step 5 - Headline KPIs

These are the big numbers at the top of any dashboard.

**✏️ Your turn:** Calculate:
- `total_sales` - sum of `Sales`
- `total_profit` - sum of `Profit`
- `profit_margin` - profit as a share of sales
- `n_orders` - number of **unique** orders (remember: rows ≠ orders!)
- `n_customers` - number of unique customers
- `avg_order_value` - sales per order

In [ ]:
# ✏️ YOUR TURN - replace every ... with code
total_sales = ...
total_profit = ...
profit_margin = ...
n_orders = ...
n_customers = ...
avg_order_value = ...

<details><summary>💡 Hint</summary>

`.sum()` and `.nunique()` are your friends. Margin and average order value are just divisions of things you already computed.

</details>

In [ ]:
check("Total sales", total_sales, 14_915_600.82)
check("Total profit", total_profit, 1_521_767.98)
check("Profit margin", profit_margin, 0.1020)
check("Orders", n_orders, 5496)
check("Customers", n_customers, 795)
check("Average order value", avg_order_value, 2713.90)

Put them in one small table - this is the *KPI summary* from the original mini-project, but formatted for humans:

In [ ]:
kpi_summary_table = pd.DataFrame({
    "KPI": ["Total Sales", "Total Profit", "Profit Margin", "Orders", "Customers", "Avg Order Value"],
    "Value": [f"${total_sales:,.0f}", f"${total_profit:,.0f}", f"{profit_margin:.1%}",
              f"{n_orders:,}", f"{n_customers:,}", f"${avg_order_value:,.0f}"],
})
kpi_summary_table

## Step 6 - KPIs by group

Totals hide the story. Break them down with `groupby` + `agg`.

**✏️ Your turn:** Build `by_category`: one row per `Product Category` with columns `Sales` (sum), `Profit` (sum)
and `Orders` (number of unique orders). Then add a `Margin` column = Profit / Sales.

In [ ]:
# ✏️ YOUR TURN - replace every ... with code
by_category = df.groupby("Product Category", observed=True).agg(
    Sales=...,
    Profit=...,
    Orders=...,
)
by_category["Margin"] = ...
by_category.sort_values("Sales", ascending=False)

<details><summary>💡 Hint</summary>

"Named aggregation": `Sales=("Sales", "sum")` means *new column Sales = sum of column Sales*.

</details>

In [ ]:
check("Furniture margin", by_category.loc["Furniture", "Margin"], 0.0227)
check("Technology profit", by_category.loc["Technology", "Profit"], 886_313.52)

🤔 **Think:** Furniture sells almost as much as Technology. Compare their margins. What would you tell the manager?

**✏️ Your turn:** Total `Sales` per `Region`, sorted from highest to lowest.

In [ ]:
# ✏️ YOUR TURN - replace every ... with code
sales_by_region = ...
sales_by_region

<details><summary>💡 Hint</summary>

`df.groupby("Region", observed=True)["Sales"].sum()` then `.sort_values(ascending=False)`.

</details>

In [ ]:
check("Top region sales (West)", sales_by_region.iloc[0], 3_597_549.27)

### Two-dimensional KPIs: pivot tables
Like the original's *class × sex* survival matrix, a pivot table shows a KPI across two dimensions at once.

**✏️ Your turn:** Make a pivot table of total `Profit` with `Product Category` as rows and `Customer Segment` as columns.

In [ ]:
# ✏️ YOUR TURN - replace every ... with code
profit_pivot = ...
profit_pivot

<details><summary>💡 Hint</summary>

`df.pivot_table(values=..., index=..., columns=..., aggfunc="sum", observed=True)`

</details>

In [ ]:
check("Corporate × Technology profit", profit_pivot.loc["Technology", "Corporate"], 374_701)

### Operational KPI: shipping speed
Does paying attention to priority actually make orders ship faster?

In [ ]:
df.groupby("Order Priority", observed=True)["Days to Ship"].mean().sort_values()

## Step 7 - Time-based KPIs

The Titanic data had no dates, so the original lesson *faked* time using ticket letters.
Our data has real order dates - so we can do proper trends.

**✏️ Your turn:**
1. `monthly_sales` - total `Sales` per `Month` (the column you created in Step 4)
2. `yearly_sales` - total `Sales` per `Year`
3. `yoy_growth` - year-over-year growth of `yearly_sales` (look up `.pct_change()`)

In [ ]:
# ✏️ YOUR TURN - replace every ... with code
monthly_sales = ...
yearly_sales = ...
yoy_growth = ...
pd.DataFrame({"Sales": yearly_sales, "YoY growth": yoy_growth})

<details><summary>💡 Hint</summary>

Same `groupby(...).sum()` pattern as before. `.pct_change()` computes (this − previous) / previous.

</details>

In [ ]:
check("Number of months", len(monthly_sales), 48)
check("2012 growth vs 2011", yoy_growth.loc[2012], 0.0824)

Monthly numbers are noisy. A **3-month rolling average** smooths them so the trend is visible:

In [ ]:
sales_trend = monthly_sales.rolling(3).mean()
sales_trend.tail()

## Step 8 - Visualize: the dashboard

The original lesson plotted one bar chart. A real dashboard combines **headline numbers** and a few **focused charts**
on one page. We use matplotlib's `subplot_mosaic` to lay out the panels.

Design choices worth learning:
- **Headline KPIs are big numbers, not charts** - a single value doesn't need an axis.
- **One color for one series.** Color only when it *means* something.
- **Horizontal bars** for category names - the labels stay readable.
- **Sorted bars** - the reader instantly sees the biggest and smallest.
- **Remove chart junk** - no top/right borders, light grid.

Read this cell slowly: every line is a design decision. Then change something and re-run it!

In [ ]:
BLUE, BLUE_LIGHT, INK, MUTED, GRID = "#2a78d6", "#9ec5f4", "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update({
    "font.size": 10, "axes.edgecolor": GRID, "axes.labelcolor": MUTED,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.titleweight": "bold",
    "axes.titlesize": 11, "axes.titlelocation": "left", "axes.titlecolor": INK,
})

fig, ax = plt.subplot_mosaic(
    [["cards", "cards"],
     ["trend", "trend"],
     ["margin", "region"]],
    figsize=(13, 10), height_ratios=[0.55, 1.4, 1.4], layout="constrained",
)
fig.patch.set_facecolor("white")
fig.suptitle("Superstore Sales KPI Dashboard · 2009–2012", x=0.01, ha="left",
             fontsize=16, fontweight="bold", color=INK)

# --- Row 1: headline KPI cards ---------------------------------------------
# One invisible axis, 4 units wide: card i lives between x=i and x=i+1.
cards = [("Total Sales", f"${total_sales/1e6:,.1f}M"),
         ("Total Profit", f"${total_profit/1e6:,.2f}M"),
         ("Profit Margin", f"{profit_margin:.1%}"),
         ("Avg Order Value", f"${avg_order_value:,.0f}")]
a = ax["cards"]
a.set_xlim(0, len(cards)); a.set_ylim(0, 1); a.set_axis_off()
for i, (label, value) in enumerate(cards):
    a.add_patch(plt.Rectangle((i + 0.03, 0), 0.94, 1, color="#f4f3f0"))
    a.text(i + 0.1, 0.68, label, fontsize=11, color=MUTED)
    a.text(i + 0.1, 0.2, value, fontsize=24, fontweight="bold", color=INK)

# --- Row 2: monthly sales trend ---------------------------------------------
a = ax["trend"]
a.plot(monthly_sales.index, monthly_sales / 1e3, color=BLUE_LIGHT, linewidth=1.5, label="Monthly sales")
a.plot(sales_trend.index, sales_trend / 1e3, color=BLUE, linewidth=2.5, label="3-month average")
a.set_title("Monthly sales ($K)")
a.legend(frameon=False, loc="upper right")

# --- Row 3 left: profit margin by category ----------------------------------
a = ax["margin"]
m = by_category["Margin"].sort_values()
bars = a.barh(m.index, m.values, color=BLUE, height=0.55)
a.bar_label(bars, labels=[f"{v:.1%}" for v in m.values], padding=4, color=INK)
a.set_title("Profit margin by category")
a.set_xlim(0, m.max() * 1.25)
a.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))

# --- Row 3 right: sales by region -------------------------------------------
a = ax["region"]
r = sales_by_region.sort_values() / 1e6
bars = a.barh(r.index, r.values, color=BLUE, height=0.6)
a.bar_label(bars, labels=[f"${v:.1f}M" for v in r.values], padding=4, color=INK)
a.set_title("Sales by region ($M)")
a.set_xlim(0, r.max() * 1.2)

for key in ["trend", "margin", "region"]:
    ax[key].spines[["top", "right"]].set_visible(False)
    ax[key].grid(axis="x" if key != "trend" else "y", color=GRID, linewidth=0.8)
    ax[key].set_axisbelow(True)

(ROOT / "images").mkdir(exist_ok=True)
fig.savefig(ROOT / "images" / "kpi_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 9 - Make it reusable

The original lesson used `input()` to filter by port. That breaks when you run the notebook automatically
(and on GitHub). A **function** is better: same logic, any filter, and testable.

The original also showed an important dashboard rule: **handle empty data gracefully** - a filter that
matches nothing must not crash the dashboard.

**✏️ Your turn:** Write `kpi_report(data)` that returns a dictionary with keys
`"Sales"`, `"Profit"`, `"Margin"`, `"Orders"`. If `data` is empty, return `None` for every value
(dividing by zero sales would otherwise break things).

In [ ]:
# ✏️ YOUR TURN - replace every ... with code
def kpi_report(data):
    if ...:
        return {"Sales": None, "Profit": None, "Margin": None, "Orders": None}
    sales = ...
    profit = ...
    return {
        "Sales": sales,
        "Profit": profit,
        "Margin": ...,
        "Orders": ...,
    }

<details><summary>💡 Hint</summary>

`data.empty` is `True` when there are no rows. The rest is copied from Step 5 - but using `data`, not `df`!

</details>

In [ ]:
check("Ontario margin", kpi_report(df[df["Region"] == "Ontario"])["Margin"], 0.1132)
check("Empty filter returns None", int(kpi_report(df[df["Region"] == "Atlantis"])["Sales"] is None), 1)

Now we can build a KPI table for **every** region in one line - a mini dashboard you could export to Excel:

In [ ]:
region_kpis = pd.DataFrame({region: kpi_report(df[df["Region"] == region])
                            for region in df["Region"].cat.categories}).T
region_kpis.sort_values("Sales", ascending=False)

## Step 10 - Insights ✍️

Numbers are only useful if someone acts on them. **✏️ Your turn:** write 3–5 bullet points in your own words.
Use the outputs above. Some questions to guide you:
- Which category earns the most *profit* - and is that the one with the most *sales*?
- About what share of order lines lose money? Is that a problem?
- Which region is strongest / weakest?
- Is the business growing? (look at year-over-year growth)
- Do "Low" priority orders ship slower?

*(Double-click this cell to edit it.)*

- Insight 1: ...
- Insight 2: ...
- Insight 3: ...

## Recap
- Loaded a real-world CSV (and handled its encoding)
- Explored it and found the row ≠ order trap
- Cleaned it: smarter missing-value filling (per group) and real date types
- Created KPI columns (`Days to Ship`, `Is Loss`, `Year`, `Month`)
- Computed headline, grouped, 2-D (pivot) and time-based KPIs
- Built and saved a dashboard image
- Wrapped the logic in a reusable, empty-safe function

### Stretch goals (to make it yours)
- Add an interactive dropdown with `ipywidgets` to pick a region and redraw the dashboard
- Turn it into a web app with **Streamlit** (`st.metric` is perfect for KPI cards)
- Find the top 10 most profitable customers and the 10 biggest loss-making products
- Is discount related to loss? Compare average `Discount` for `Is Loss == True` vs `False`

---


> **Credit:** This project is inspired by the free lesson
> [*Building a KPI Dashboard with Pandas*](https://www.mathewkanalytics.com/lessons/mastering-pandas/058-building-a-kpi-dashboard-with-pandas/)
> by **[Mathew K Analytics](https://www.mathewkanalytics.com/)**
> ([YouTube video](https://www.youtube.com/watch?v=lid1q679cs0)).
> The original uses the Titanic dataset; this version applies the same workflow to a retail sales dataset and extends it.
>
> Dataset: *Superstore Sales* sample data, via [curran/data](https://github.com/curran/data/tree/gh-pages/superstoreSales).